In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import sys
import os
import logging
import time
import json
from datetime import datetime
import re
import subprocess


In [2]:
spark = SparkSession.builder \
    .appName("DataProcessingApp") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/16 17:29:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


* **header=True:** Treats the first row as column names.
* **inferSchema=True:** Automatically detects data types.
* **Efficient for loading structured data into a distributed format.**

In [3]:
dfcsv = spark.read.csv("/kaggle/input/human-resources-data-set/HRDataset_v14.csv", header=True, inferSchema=True)
dfcsv.show()

25/09/16 17:30:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-----+---------+---------------+--------+-----------+------+-----------+----------------------+------+-----+----------+--------------------+-----+-----+--------+---+-----------+-------------------+--------------+--------------------+----------+-----------------+-----------------+--------------------+--------------------+---------------+---------+--------------------+----------------+----------------+---------------+--------------------+--------------------------+--------------+--------+
|       Employee_Name|EmpID|MarriedID|MaritalStatusID|GenderID|EmpStatusID|DeptID|PerfScoreID|FromDiversityJobFairID|Salary|Termd|PositionID|            Position|State|  Zip|     DOB|Sex|MaritalDesc|        CitizenDesc|HispanicLatino|            RaceDesc|DateofHire|DateofTermination|       TermReason|    EmploymentStatus|          Department|    ManagerName|ManagerID|   RecruitmentSource|PerformanceScore|EngagementSurvey|EmpSatisfaction|SpecialProjectsCount|LastPerformanceReview_Da

**Printing DataFrame Schema** 
_To view the structure (columns and data types):_

In [4]:
dfcsv.printSchema()

root
 |-- Employee_Name: string (nullable = true)
 |-- EmpID: integer (nullable = true)
 |-- MarriedID: integer (nullable = true)
 |-- MaritalStatusID: integer (nullable = true)
 |-- GenderID: integer (nullable = true)
 |-- EmpStatusID: integer (nullable = true)
 |-- DeptID: integer (nullable = true)
 |-- PerfScoreID: integer (nullable = true)
 |-- FromDiversityJobFairID: integer (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Termd: integer (nullable = true)
 |-- PositionID: integer (nullable = true)
 |-- Position: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zip: integer (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- MaritalDesc: string (nullable = true)
 |-- CitizenDesc: string (nullable = true)
 |-- HispanicLatino: string (nullable = true)
 |-- RaceDesc: string (nullable = true)
 |-- DateofHire: string (nullable = true)
 |-- DateofTermination: string (nullable = true)
 |-- TermReason: string (nullable

**Common tasks:** 
* Row count
* Group and aggregate:

In [5]:
from pyspark.sql import functions as F

totalcount = dfcsv.count()

dfcsv.groupBy("GenderID").agg({"Salary": "sum", "Salary": "avg"}).show()


dfcsv.groupBy("GenderID").agg(
    F.sum("Salary").alias("Salary_Sum"),
    F.avg("Salary").alias("Salary_Avg")
).show()

+--------+-----------------+
|GenderID|      avg(Salary)|
+--------+-----------------+
|       1|          70629.4|
|       0|67786.72727272728|
+--------+-----------------+

+--------+----------+-----------------+
|GenderID|Salary_Sum|       Salary_Avg|
+--------+----------+-----------------+
|       1|   9534969|          70629.4|
|       0|  11930464|67786.72727272728|
+--------+----------+-----------------+



**Key DataFrame methods:**
| *Function* |*Use Case* | *Equivalent in SQL* |
| ---- |---- | ---- |
| select() | Choose specific columns | SELECT |
| filter() | Filter rows by condition | WHERE |
| groupBy() | Group rows by one or more columns | GROUP BY |
| agg() | Aggregate with a function like sum, avg | AGGREGATE |

**SQL:** _Select Employee_Name, HispanicLatino from csv_file where State = 'MA'_

In [6]:
dfcsv.filter(dfcsv.State == "MA").select("Employee_Name", "HispanicLatino").show() 

+--------------------+--------------+
|       Employee_Name|HispanicLatino|
+--------------------+--------------+
| Adinolfi, Wilson  K|            No|
|Ait Sidi, Karthik...|            No|
|   Akinkuolie, Sarah|            No|
|        Alagbe,Trina|            No|
|    Anderson, Carol |            No|
|   Anderson, Linda  |            No|
|     Andreola, Colby|            No|
|         Athwal, Sam|            No|
|    Bachiochi, Linda|            No|
|  Bacong, Alejandro |            No|
|Baczenski, Rachael  |           Yes|
|     Barbara, Thomas|           Yes|
|Barone, Francesco  A|            No|
|       Barton, Nader|            No|
|       Bates, Norman|            No|
|    Beak, Kimberly  |            No|
| Beatrice, Courtney |            No|
|       Becker, Renee|           Yes|
|       Becker, Scott|            No|
|     Bernstein, Sean|           Yes|
+--------------------+--------------+
only showing top 20 rows



*To get employees whose salary is 40% above the average salary, it need to:*
1. 	Calculate the total salary across all employees.
2. 	Compute the threshold (i.e., total salary × 1.4).
3. 	Filter employees whose salary exceeds that threshold.

In [7]:
from pyspark.sql import functions as F

# Step 1: Calculate average salary
avg_sal = dfcsv.agg(F.avg("Salary").alias("Avg_Salary")).collect()[0]["Avg_Salary"]

print(f"Average Salary: {avg_sal}")

# Step 2: Compute 40% above total salary
threshold = avg_sal * 1.4

# Step 3: Filter employees whose salary exceeds the threshold
dfcsv.filter(dfcsv.Salary > threshold) \
     .withColumn("Avg_Salary", F.lit(avg_sal)) \
     .select("Employee_Name", "Salary", "Avg_Salary") \
     .show()


Average Salary: 69020.6848874598
+--------------------+------+----------------+
|       Employee_Name|Salary|      Avg_Salary|
+--------------------+------+----------------+
|Ait Sidi, Karthik...|104437|69020.6848874598|
|       Becker, Renee|110000|69020.6848874598|
|        Booth, Frank|103613|69020.6848874598|
|   Boutwell, Bonalyn|106367|69020.6848874598|
|    Carr, Claudia  N|100031|69020.6848874598|
|   Champaigne, Brian|110929|69020.6848874598|
|      Corleone, Vito|170500|69020.6848874598|
|   Del Bosque, Keyla|101199|69020.6848874598|
|       DeVito, Tommy| 96820|69020.6848874598|
|       Dougall, Eric|138888|69020.6848874598|
|      Exantus, Susan| 99280|69020.6848874598|
|         Foss, Jason|178000|69020.6848874598|
|   Foster-Baker, Amy| 99351|69020.6848874598|
|       Goble, Taisha|114800|69020.6848874598|
|        Gruber, Hans| 99020|69020.6848874598|
|       Horton, Jayne| 97999|69020.6848874598|
|     Houlihan, Debra|180000|69020.6848874598|
|    Johnson, Noelle |10570

*Count the total number of rows*

In [8]:
row_count = dfcsv.count()
print(f"Total rows: {row_count}")

Total rows: 311


*Applying more than one filter*

In [9]:
# Option 1: Chaining .filter() Calls
dfcsv.filter((col("GenderID") == 1) & (col("Salary") > 60000)).show(5)

# Option 2: Single .filter() with Logical Operators
# Use  for & AND,  | for OR, and  ~ for NOT.
dfcsv.filter((col("State") == "MA") & (col("Salary") > 60000)).show(5)

# Option 3: Using .where() Method
dfcsv.where((col("MarriedID") == 0) & (col("Salary") > 70000)).show(5)

#Filter + Select
dfcsv.filter((col("GenderID") == 1) & (col("Salary") > 60000)) \
     .select("Employee_Name", "Salary", "GenderID") \
     .show(5)

+--------------------+-----+---------+---------------+--------+-----------+------+-----------+----------------------+------+-----+----------+--------------------+-----+-----+--------+---+-----------+-----------+--------------+--------------------+----------+-----------------+--------------------+--------------------+-----------------+--------------+---------+------------------+----------------+----------------+---------------+--------------------+--------------------------+--------------+--------+
|       Employee_Name|EmpID|MarriedID|MaritalStatusID|GenderID|EmpStatusID|DeptID|PerfScoreID|FromDiversityJobFairID|Salary|Termd|PositionID|            Position|State|  Zip|     DOB|Sex|MaritalDesc|CitizenDesc|HispanicLatino|            RaceDesc|DateofHire|DateofTermination|          TermReason|    EmploymentStatus|       Department|   ManagerName|ManagerID| RecruitmentSource|PerformanceScore|EngagementSurvey|EmpSatisfaction|SpecialProjectsCount|LastPerformanceReview_Date|DaysLateLast30|Abse

>#### Schema Inference and Manual Schema Definition:
>>Spark can automatically infer schemas, but it might misinterpret data types, particularly with complex or ambiguous data. Manually defining a schema can ensure accurate data handling.

>#### DataTypes in PySpark DataFrames 
>>To manually configure a schema, we define the datatype using the StructField function, calling the appropriate datatype method. PySpark DataFrames support various data types, similar to SQL and Pandas.

In [10]:
# Show Datatypes
# Option 1: Use .printSchema() for a Tree View

#dfcsv.printSchema()

# Option 2: Use .dtypes for a List of Tuples
dfcsv.dtypes

# Option 3: Use .schema for a Detailed Schema Object
dfcsv.schema

# OR Convert to Pandas for Quick Profiling
# If you want to do a quick audit or export to Pandas for profiling, you can convert a sample to Pandas:
dfcsv.limit(100).toPandas().info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Employee_Name               100 non-null    object 
 1   EmpID                       100 non-null    int32  
 2   MarriedID                   100 non-null    int32  
 3   MaritalStatusID             100 non-null    int32  
 4   GenderID                    100 non-null    int32  
 5   EmpStatusID                 100 non-null    int32  
 6   DeptID                      100 non-null    int32  
 7   PerfScoreID                 100 non-null    int32  
 8   FromDiversityJobFairID      100 non-null    int32  
 9   Salary                      100 non-null    int32  
 10  Termd                       100 non-null    int32  
 11  PositionID                  100 non-null    int32  
 12  Position                    100 non-null    object 
 13  State                       100 non-

### Display list of Columns

In [11]:
# Option 1: Use  to List All Column Names
dfcsv.columns
# Option 2: Use .select("*").show() with truncate=False
dfcsv.select("*").show(truncate=False, n=5)
# Option 3: Convert to Pandas (for Small Datasets)
dfcsv.limit(5).toPandas().head()
# Option 4: Use .take() for a List of Rows
dfcsv.take(5)
# Print Schema for Column Names + Types
dfcsv.printSchema()

+------------------------+-----+---------+---------------+--------+-----------+------+-----------+----------------------+------+-----+----------+------------------------+-----+----+--------+---+-----------+-----------+--------------+--------+----------+-----------------+-----------------+----------------------+-----------------+--------------+---------+-----------------+----------------+----------------+---------------+--------------------+--------------------------+--------------+--------+
|Employee_Name           |EmpID|MarriedID|MaritalStatusID|GenderID|EmpStatusID|DeptID|PerfScoreID|FromDiversityJobFairID|Salary|Termd|PositionID|Position                |State|Zip |DOB     |Sex|MaritalDesc|CitizenDesc|HispanicLatino|RaceDesc|DateofHire|DateofTermination|TermReason       |EmploymentStatus      |Department       |ManagerName   |ManagerID|RecruitmentSource|PerformanceScore|EngagementSurvey|EmpSatisfaction|SpecialProjectsCount|LastPerformanceReview_Date|DaysLateLast30|Absences|
+-------

#### Finding NULL or NOT NULL
>
>> In PySpark, filtering for NULL values is just as intuitive as SQL’s WHERE column IS NULL; use the .isNull() method on the column.


In [19]:
# Filter Rows Where a Column Is NULL
# SQL : SELECT * FROM dfcsv WHERE DateofTermination IS NULL;
dfcsv.filter(dfcsv.DateofTermination.isNull()).show(5)

# Filter Rows Where a Column Is NOT NULL
# SQL : SELECT * FROM dfcsv WHERE DateofTermination IS NOT NULL;
dfcsv.filter(dfcsv.DateofTermination.isNotNull()).show(5)

# Combine with Other Filters
# WHERE DateofTermination IS NULL AND State = 'MA'.
dfcsv.filter((dfcsv.DateofTermination.isNull()) & (dfcsv.State == "MA")).show(7)


+-------------------+-----+---------+---------------+--------+-----------+------+-----------+----------------------+------+-----+----------+--------------------+-----+----+--------+---+-----------+-----------+--------------+--------+----------+-----------------+-----------------+----------------+--------------------+---------------+---------+-----------------+----------------+----------------+---------------+--------------------+--------------------------+--------------+--------+
|      Employee_Name|EmpID|MarriedID|MaritalStatusID|GenderID|EmpStatusID|DeptID|PerfScoreID|FromDiversityJobFairID|Salary|Termd|PositionID|            Position|State| Zip|     DOB|Sex|MaritalDesc|CitizenDesc|HispanicLatino|RaceDesc|DateofHire|DateofTermination|       TermReason|EmploymentStatus|          Department|    ManagerName|ManagerID|RecruitmentSource|PerformanceScore|EngagementSurvey|EmpSatisfaction|SpecialProjectsCount|LastPerformanceReview_Date|DaysLateLast30|Absences|
+-------------------+-----+---

### Handling Missing Data
**Handling null values is crucial in data analysis as missing data can lead to skewed results or errors during processing. PySpark provides two primary methods for managing missing values:**
>
>> #### Dropping Null Values:
>>
>>> * The .na.drop() method can be used to drop rows that contain null values. 
>>> * This can be done across the entire DataFrame or for specific columns. 
>>> * While this method simplifies the dataset, it might significantly reduce the dataset size if null values are common, which could lead to loss of important data.